In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

LAG, ROLL, MIN_PERIODS, TAIL = 21, 7, 3, 14
CFR_WIN, DBL_WIN, PEAK_K, PEAK_FLOOR_PM = 28, 14, 21, 50.0
ISO = ["ISR", "GBR", "USA", "BRA", "IND", "ZAF", "DEU"]

In [4]:
try:
    df = pd.read_parquet("covid_clean.parquet")  
    src = "parquet"
except FileNotFoundError:
    raw = pd.read_csv("data/owid-covid-data.csv")
    raw = raw[~raw["iso_code"].str.startswith("OWID_", na=False)]  # 395,311 real rows
    df = raw[raw["iso_code"].isin(ISO)].copy()  # -> your 11,726
    src = "csv-rebuild"

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).sort_values(["iso_code", "date"]).reset_index(drop=True)

BASE = [
    "new_cases",
    "new_deaths",
    "new_cases_smoothed",
    "new_deaths_smoothed",
    "new_cases_per_million",
    "new_deaths_per_million",
    "new_cases_smoothed_per_million",
    "new_deaths_smoothed_per_million",
    "people_fully_vaccinated_per_hundred",
    "population",
    "median_age",
    "hospital_beds_per_thousand",
    "gdp_per_capita",
    "life_expectancy",
]
for c in BASE:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

In [5]:
sizes = df.groupby("iso_code").size()
if sizes.nunique() != 1:  # gappy -> your current 11,726 case
    cutoff = df["date"].max() - pd.Timedelta(days=TAIL)
    df = df[df["date"] <= cutoff].copy()
    full = pd.date_range(df["date"].min(), cutoff, freq="D")
    df = (
        df.set_index(["iso_code", "date"])
        .reindex(
            pd.MultiIndex.from_product(
                [sorted(df["iso_code"].unique()), full], names=["iso_code", "date"]
            )
        )
        .sort_index()
        .reset_index()
    )
    STATIC = [
        "location",
        "continent",
        "population",
        "median_age",
        "hospital_beds_per_thousand",
        "gdp_per_capita",
        "life_expectancy",
    ]
    STATIC = [s for s in STATIC if s in df.columns]
    df[STATIC] = df.groupby("iso_code")[STATIC].ffill()
    print(
        f"[0] rebuilt calendar-safe frame from {src}: trimmed TAIL={TAIL}, "
        f"reindexed to {len(full)} days x {df['iso_code'].nunique()} countries"
    )
else:
    cutoff = df["date"].max()
    print(
        f"[0] loaded calendar-safe frame from {src}: {sizes.iloc[0]} days/country (rectangle OK)"
    )

assert (
    df.groupby("iso_code").size().nunique() == 1
), "reindex failed -> row counts differ"
assert not df.duplicated(
    ["iso_code", "date"]
).any(), "panel not unique per (country,date)"

[0] loaded calendar-safe frame from parquet: 1668 days/country (rectangle OK)


## 7-DAY ROLLS from base

In [6]:
def roll7(s):
    return s.rolling(ROLL, min_periods=MIN_PERIODS).mean()


for c in [
    "new_cases",
    "new_deaths",
    "new_cases_per_million",
    "new_deaths_per_million",
    "new_cases_smoothed_per_million",
    "new_deaths_smoothed_per_million",
    "people_fully_vaccinated_per_hundred",
]:
    if c in df.columns:
        df[f"{c}_7d"] = df.groupby("iso_code")[c].transform(roll7)

## 2. LAGGED vs NAIVE CFR 

In [7]:
d28 = df.groupby("iso_code")["new_deaths"].transform(lambda s: s.rolling(CFR_WIN).sum())
c28_now = df.groupby("iso_code")["new_cases"].transform(
    lambda s: s.rolling(CFR_WIN).sum()
)
c28_lag = c28_now.groupby(df["iso_code"]).shift(LAG)  # cases from LAG days ago
df["cfr_lagged_28d"] = d28 / c28_lag.replace(0, np.nan)  # RIGHT one (headline)
df["cfr_naive_28d"] = d28 / c28_now.replace(0, np.nan)  # same window, NO lag
df["cfr_cumulative"] = (
    df["total_deaths"] / df["total_cases"].replace(0, np.nan)
    if "total_deaths" in df
    else np.nan
)

In [8]:
val = df[
    ["iso_code", "date", "new_deaths_per_million_7d", "new_cases_per_million_7d"]
].dropna()
rows = []
for L in [0, 7, 14, 21, 28, 35]:
    shifted = val.groupby("iso_code")["new_cases_per_million_7d"].shift(L)
    rows.append((L, np.corrcoef(val["new_deaths_per_million_7d"], shifted)[0, 1]))
lag_diag = pd.DataFrame(
    rows, columns=["lag_days", "pearson_r(death7d ~ case7d_shifted)"]
)
print(
    "\n[2b] lag-direction diagnostic (r should PEAK near LAG=21; if it peaks at 0, "
    "your shift sign is wrong):"
)
print(lag_diag.round(3).to_string(index=False))


[2b] lag-direction diagnostic (r should PEAK near LAG=21; if it peaks at 0, your shift sign is wrong):
 lag_days  pearson_r(death7d ~ case7d_shifted)
        0                                0.283
        7                                  NaN
       14                                  NaN
       21                                  NaN
       28                                  NaN
       35                                  NaN


## 3. ROLLING DOUBLING TIME from log-linear slope of smoothed cases

In [9]:
def doubling_time(s, win=DBL_WIN):
    y = np.log(np.clip(s.values, 1e-6, None))
    x = np.arange(len(s))
    out = np.full(len(s), np.nan)
    for i in range(win, len(s)):
        r = np.polyfit(x[i - win : i], y[i - win : i], 1)[0]  # daily growth rate
        out[i] = np.log(2) / r if r > 1e-4 else np.nan  # nan when flat/declining
    return pd.Series(out, index=s.index)


df["doubling_time_14d"] = df.groupby("iso_code", group_keys=False)[
    "new_cases_smoothed_per_million_7d"
].apply(doubling_time)

## 4. WAVE PEAKS by documented rule

In [10]:
def flag_peaks(s, k=PEAK_K, floor=PEAK_FLOOR_PM):
    rmax = s.rolling(2 * k + 1, center=True, min_periods=1).max()
    return ((s == rmax) & (s >= floor)).astype(int)


df["is_wave_peak"] = df.groupby("iso_code", group_keys=False)[
    "new_cases_smoothed_per_million_7d"
].apply(flag_peaks)
print(
    "\n[4] wave peaks detected per country (smoothed cases/million, local max +/-21d, floor 50/m):"
)
print(df.groupby("iso_code")["is_wave_peak"].sum().to_string())


[4] wave peaks detected per country (smoothed cases/million, local max +/-21d, floor 50/m):
iso_code
BRA     9
DEU    11
GBR    12
IND     3
ISR    11
USA     9
ZAF     6


## 5. ROLLOUT SPEED scalar 

In [11]:
def rollout_speed(g):
    s = g.set_index("date")["people_fully_vaccinated_per_hundred"].sort_index().dropna()
    try:
        t10 = s[s >= 10].index[0]
        t50 = s[s >= 50].index[0]
        return pd.Series({"days_10_to_50": (t50 - t10).days, "date_50pct": t50})
    except IndexError:
        return pd.Series({"days_10_to_50": np.nan, "date_50pct": pd.NaT})


speed = df.groupby("iso_code").apply(rollout_speed, include_groups=False)

## 6. POST-VAX DEATH BURDEN

In [12]:
def post_vax_peak(g, t50):
    if pd.isna(t50):
        return np.nan
    s = g.set_index("date")["new_deaths_per_million_7d"].dropna()
    s = s[s.index >= t50]
    return s.max() if len(s) else np.nan


post = df.merge(speed["date_50pct"], on="iso_code", how="left")
post_peak = (
    post.groupby("iso_code")
    .apply(lambda g: post_vax_peak(g, g["date_50pct"].iloc[0]), include_groups=False)
    .rename("peak_deaths_pm_post50")
)

## 7. CONFOUNDER TABLE

In [13]:
conf = df.groupby("iso_code").agg(
    location=("location", "last"),
    continent=("continent", "last"),
    median_age=("median_age", "last"),
    hosp_beds_k=("hospital_beds_per_thousand", "last"),
    gdp_ppp=("gdp_per_capita", "last"),
    life_exp=("life_expectancy", "last"),
)
country_table = pd.concat([conf, speed[["days_10_to_50"]], post_peak], axis=1)

# excess-mortality coverage probe (the Day-1 robustness idea — check usability FIRST)
if "excess_mortality_cumulative_per_million" in df.columns:
    em = (
        df.groupby("iso_code")["excess_mortality_cumulative_per_million"]
        .apply(lambda s: s.notna().mean() * 100)
        .rename("excess_mort_coverage_%")
    )
    country_table = country_table.join(em)
print(
    "\n[7] country table (the focused-question inputs; values are YOURS, not invented):"
)
print(country_table.round(2).to_string())

df.to_parquet("covid_derived.parquet", index=False)
country_table.to_csv("country_rollout_vs_death.csv")
print(f"\nsaved covid_derived.parquet {df.shape} + country_rollout_vs_death.csv")


[7] country table (the focused-question inputs; values are YOURS, not invented):
                location      continent  median_age  hosp_beds_k   gdp_ppp  life_exp  days_10_to_50  peak_deaths_pm_post50  excess_mort_coverage_%
iso_code                                                                                                                                          
BRA               Brazil  South America        33.5         2.20  14103.45     75.88          148.0                   4.52                    2.88
DEU              Germany         Europe        46.6         8.00  45229.24     81.33           76.0                   5.25                   12.53
GBR       United Kingdom         Europe        40.8         2.54  39753.24     81.32           87.0                   3.46                   12.53
IND                India           Asia        28.2         0.53   6426.67     69.66          156.0                   0.79                    0.00
ISR               Israel           A